# Example: Using QR iteration to compute eigenvalues and eigenvectors
This example will familiarize students with computing the [eigenvalues and eigenvectors]() of a real matrix $\mathbf{A}\in\mathbb{R}^{n\times{n}}$ using [the QR iteration algorithm](https://en.wikipedia.org/wiki/QR_algorithm).

__Learning objectives:__ Students will learn to implement and compare the QR iteration algorithm for eigenvalue computation with Julia's built-in eigenvalue solver. Through hands-on implementation and benchmarking, students will understand the trade-offs between custom algorithm development and using optimized library functions.

Let's go!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's setup our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

___

## Task 1: Compute eigenvalues and eigenvectors
Let's do an example where we compute the eigenvalues and eigenvectors of a square matrix $\mathbf{A}$ using our implementation of the [QR iteration algorithm](https://en.wikipedia.org/wiki/QR_algorithm), which is implemented in the [`qriteration(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.qriteration). 

Compute the eigenvalues and eigenvectors of the matrix:
$$
\mathbf{A} = \begin{bmatrix}
3.0 & -0.3 & -0.2 \\
0.1 & 7.0 & -0.3 \\
0.3 & -0.2 & 10.0 \\
\end{bmatrix}
$$

How well does our answer compare to the values generated by [eigen function which is part of the Julia standard library?](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen)

In [2]:
A = [3.0 -0.3 -0.2 ; 0.1 7.0 -0.3 ; 0.3 -0.2 10.0] # Setup the n x n matrix A (n = 3)

3×3 Matrix{Float64}:
 3.0  -0.3  -0.2
 0.1   7.0  -0.3
 0.3  -0.2  10.0

First, let's use the [built-in `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) and see what we get. The [`eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) takes a square matrix `A` as an argument and returns the eigendecomposition.

The decomposition is stored in the [Eigen factorization object `F`](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.Eigen) which contains the eigenvalues and eigenvectors.
> __What's in `F`?__  The eigenvalues can be obtained via`F.values` and the eigenvectors as the columns of the matrix `F.vectors`. (The kth eigenvector can be obtained from the slice `F.vectors[:, k].`)

What do we get back?

In [3]:
# Decompose using the built-in function
F = eigen(A);   # eigenvalues and vectors in F of type Eigen
λ = F.values;   # vector of eigenvalues
V = F.vectors;  # 3 x 3 matrix of eigenvectors, each col is an eigenvector

Eigenvalues (obtained from `F.values`):

In [4]:
λ

3-element Vector{Float64}:
  3.017277143754452
  6.9699146114784165
 10.012808244767134

Eigenvectors (obtained from `F.vectors`):

In [5]:
V

3×3 Matrix{Float64}:
 -0.998641    0.0788277  -0.024097
  0.0283674  -0.994181   -0.099848
  0.0437173  -0.0734251   0.994711

Next, let's call [the `qriteration(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.qriteration) and see what happens. 

> __Fun activity__: Try modifying the `maxiter` parameter or the `tolerance` parameter to see how it affects the convergence of the algorithm. We expect that increasing the `maxiter` will allow the algorithm more iterations to converge, while decreasing the `tolerance` will make the algorithm more strict about convergence.

What do we get (we'll save our results to the $\hat{\lambda}$ and $\hat{\mathbf{V}}$ variables)?

In [ ]:
(λ̂,V̂) = qriteration(A; maxiter=1000, tolerance=1e-8); # Call our qriteration function

#### Check: Do we get the same values?
Let's compare our computed values with the values calculated using the builtin function using the [norm function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.norm) exported by [the `LinearAlgebra.jl` package included with the standard library](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#man-linalg).

> __Vector norm__
>
> A function $|\cdot|:\mathbb{K}^n!\to\;[0,\infty)$ ($\mathbb{K}=\mathbb{R}$ or $\mathbb{C}$) is a **norm** if, for all $u,v\in\mathbb{K}^n$ and $\alpha\in\mathbb{K}$,
>
> 1. **Definiteness:** $|v|\ge 0$ and $|v|=0 \iff v=0$.
> 2. **Homogeneity:** $|\alpha v|=|\alpha|\times|v|$.
> 3. **Triangle inequality:** $|u+v|\le |u|+|v|$.

Here, we will use a particular norm, the **2-norm** (or **Euclidean norm**), which is an example of a p-norm:

> **Definition ($p$-norm / $\ell_p$ norm).**
> 
> Let $\mathbb{K}\in{\mathbb{R},\mathbb{C}}$ and $x=(x_1,\ldots,x_n)\in\mathbb{K}^n$. Then, for $1\le p<\infty$ the p-norm is given by:
> $$
> \|x\|_{p}=\Big(\sum_{i=1}^n |x_i|^{\,p}\Big)^{1/p}.
> $$
> and for $p=\infty$:
> $$
> \|x\|_{\infty}=\max_{1\le i\le n}|x_i|.
> $$

Let's check if the eigenvalues computed by the [`eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) are the same as the ones we just calculated [by our `qriteration(...)` implementation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.qriteration).

In [7]:
[λ λ̂] # eigen is the first col, our is the second col

3×2 Matrix{Float64}:
  3.01728   3.01728
  6.96991   6.96991
 10.0128   10.0128

Compute the 2-norm of the difference between the two sets of eigenvalues (similar values should yield a small norm).

In [8]:
norm(λ - λ̂) # small means two vectors are close!

2.2554654274383627e-6

Alternatively, if $\hat v$ is computed and $v^\star$ is the reference, we can also use the **absolute error** $|\hat v-v^{\star}|$ and, when $v^\star\neq 0$, the **relative error** $|\hat v-v^\star|/|v^\star|$ to measure how close $\hat v$ is to $v^\star$.

Let's use the [isapprox function](https://docs.julialang.org/en/v1/base/math/#Base.isapprox) to check if $\lambda$ and $\hat{\lambda}$ are close in some relative (`rtol`) or absolute (`atol`) sense (we check every value using the `.` vectorized syntax):

In [9]:
isapprox.(λ̂,λ, atol=1e-5)

3-element BitVector:
 1
 1
 1

Let's do the same thing with the eigenvectors. How similar are our eigenvectors computed using [the `qriteration(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.qriteration) to those calculated using [the `eigen(...)` method](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen)?

Let's compare the eigenvectors for the largest eigenvalue.


In [17]:
let
    i = argmax(λ) # which eigenvalue do I want to check? (returns index of largest value)
    ϵ = norm(V[:,i] - V̂[i]); # compute the norm of the difference
    println("Eigenvector $(i) norm difference: ", ϵ) # small means two vectors are close!
end

Eigenvector 3 norm difference: 5.278786035605725e-8


So, we are close to the true eigenvalues and eigenvectors! But how well do we perform in terms of memory and computational efficiency?
___

## Task 2: So we get the correct answer. But... buy versus build?
In this task, we'll compare the performance of our QR iteration implementation to the built-in eigen function.

> __True backstory__: I was supposed to edit a paper for a PhD student in my lab (an essential part of his thesis). I said I would do it over the weekend. However, instead I spent almost the entire weekend writing [the `qriteration(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/solvers/#VLDataScienceMachineLearningPackage.qriteration). Was that a valuable use of my time?

Fast forward to Monday morning at the group meeting, the student excitedly asks me about the edits, and I have to explain that I got distracted by writing an implementation of the [QR iteration algorithm](https://en.wikipedia.org/wiki/QR_algorithm).

> __Was it worth it?__ Does our implementation of the [QR iteration algorithm](https://en.wikipedia.org/wiki/QR_algorithm) for computing eigenvalues and eigenvectors beat (in either time or memory) the [built-in eigen function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen)? Let's check this using [the `@benchmark` macro exported by the `BenchmarkTools.jl` package](https://docs.julialang.org/en/v1/stdlib/Benchmark/#Benchmark.@benchmark).

First, let's check [the built-in eigen function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen).

In [11]:
@benchmark eigen($A)

BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.554 μs … 745.821 μs  ┊ GC (min … max):  0.00% … 99.38%
 Time  (median):     1.800 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   2.052 μs ±  12.132 μs  ┊ GC (mean ± σ):  10.17% ±  1.72%

   ▁  ▂▃▃▃▃▅▇█████▇▇▆▅▅▄▄▃▂▃▂▂▁ ▁▁   ▁                        ▃
  ▆████████████████████████████████████▇██▇▇█▇▇▇▇█▇▇██▇█▇▇▆▆▆ █
  1.55 μs      Histogram: log(frequency) by time      2.54 μs <

 Memory estimate: 4.53 KiB, allocs estimate: 26.

Now, we'll check the performance of our implementation using [the `@benchmark` macro](https://github.com/JuliaCI/BenchmarkTools.jl).

In [12]:
@benchmark qriteration($A)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  13.000 μs …  4.486 ms  ┊ GC (min … max): 0.00% … 99.09%
 Time  (median):     13.541 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   14.332 μs ± 44.743 μs  ┊ GC (mean ± σ):  3.10% ±  0.99%

    ▃▃█                                                        
  ▁▂████▆▆▃▅▄▃▃▂▂▁▁▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  13 μs           Histogram: frequency by time        18.9 μs <

 Memory estimate: 16.16 KiB, allocs estimate: 269.

Our implementation is worse than the built-in eigen function in both time and memory usage. So, now spending the weekend implementing the QR iteration algorithm seems like a waste of time. It was fun(ish) to implement, but the performance gains just aren't there. 

You should (almost) always buy versus build!
___

## Summary
In this notebook, we explored the QR iteration algorithm for computing eigenvalues and eigenvectors, comparing our custom implementation against Julia's highly optimized built-in `eigen(...)` function. 

While our implementation successfully computed accurate eigenvalues and eigenvectors (as verified by norm comparisons and relative error checks), the benchmarking results revealed that it significantly underperformed the standard library function in both execution time and memory efficiency. 

This exercise perfectly illustrates (yet again) the classic buy versus build dilemma in scientific computing, demonstrating that while implementing algorithms from scratch provides valuable learning insights, production code should almost always leverage well-tested, optimized library functions unless there are compelling reasons to do otherwise.